# Step 3.1: Architectural Strategy and Environment Setup

## 1. Revised Augmentation Strategy

This second iteration explicitly removes the spatial data augmentation pipeline and Mixup for regularisation.

As established in the first version, the baseline architecture does not exhibit overfitting on the raw dataset. Removing regularization aims to prevent the introduction of unnecessary noise, allowing the model to converge more effectively on the underlying artistic features.

## 2. Hardware Optimisation

Implements `set_memory_growth` to prevent TensorFlow from allocating the entirety of the VRAM at startup, avoiding hard crashes during execution, and enables Accelerated Linear Algebra (XLA) Just-In-Time compilation to fuse TensorFlow operations into optimised GPU kernels, significantly accelerating batch processing after the initial compilation overhead.

In [ ]:
import os
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
from keras import Model, layers
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")


# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']
XLA JIT enabled.


# Step 3.2: Custom Convolutional Neural Network Design

## 1. Residual Block Implementation

Defines a custom `ResidualBlock` by subclassing `keras.layers.Layer`, ensuring Keras correctly tracks all internal weights and biases, and incorporates a 1x1 convolutional projection with Batch Normalisation on the shortcut path when strides exceed 1, allowing for perfect gradient flow while matching spatial dimensions.

In [3]:
class ResidualBlock(layers.Layer):
    """
    Single residual block: Conv → BN → Activation + shortcut projection.

    Storing conv/bn/activation as named attributes of a Layer subclass
    guarantees Keras tracks their weights correctly.
    The original bug stored these inside plain Python dicts inside a plain
    Python list — Keras never registered them, so they were never trained.
    """

    def __init__(self, filters, kernel_size, stride, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.stride      = stride
        self.activation  = activation

        # He normal: correct initialisation for ReLU networks
        init = "he_normal"

        self.conv     = layers.Conv2D(filters, kernel_size, strides=stride,
                                      padding="same", use_bias=False,
                                      kernel_initializer=init)
        self.bn       = layers.BatchNormalization(momentum=0.9)
        self.actv     = layers.Activation(activation)
        self.shortcut_conv = layers.Conv2D(filters, (1, 1), strides=stride,
                                           padding="same", use_bias=False,
                                           kernel_initializer=init)
        self.shortcut_bn   = layers.BatchNormalization(momentum=0.9)  # BN on shortcut too
        self.add       = layers.Add()

    def call(self, x, training=False):
        skip = self.shortcut_conv(x)
        skip = self.shortcut_bn(skip, training=training)  # normalise shortcut

        x = self.conv(x)
        x = self.bn(x, training=training)
        # Add FIRST, then activate — standard post-activation ResNet
        x = self.add([x, skip])
        return self.actv(x)

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters, "kernel_size": self.kernel_size,
                "stride": self.stride,   "activation": self.activation}

## 2. Main CNN Architecture (`MyCNN`)

Integrates a `layers.Rescaling(1./255)` layer directly into the model definition to ensure compatibility with transfer learning pipelines and standardise input tensors.
Also, it builds the feature extraction backbone using a provided list of configurations (`conv_configs`), followed by Global Average Pooling and a dynamically generated dense classification head with Dropout for regularisation.

In [4]:
class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        # Store as a Python list of Layer objects assigned to self.
        # Keras DOES track a list of Layers set as an attribute via __setattr__,
        # as long as the list itself is set at attribute assignment time (not grown later).
        # Safest pattern: build the full list first, then assign once.
        self.blocks = [
            ResidualBlock(f, k, s, activation=activation,
                          name=f"block_{i}")
            for i, (f, k, s) in enumerate(conv_configs)
        ]

        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        dense_list = []
        for i, u in enumerate(self.dense_configs):
            dense_list.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            dense_list.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.dense_layers = dense_list
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtém a configuração base da superclasse
        config = super().get_config()
        # Adiciona os teus argumentos personalizados ao dicionário
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        x = self.rescaling(inputs)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.gap(x)
        for layer in self.dense_layers:
            # Dropout needs training flag; Dense does not
            x = layer(x, training=training) if isinstance(layer, layers.Dropout) else layer(x)
        return self.classifier(x)

# Step 3.3: Hyperparameters and Data Pipeline

## 1. Global Configuration

Establishes critical training parameters, notably reducing the batch size to 16 to accommodate the larger 384x384 image resolution within an 8GB VRAM constraint, and dynamically generates `Checkpoints` and `Metrics` directories to store model artifacts and training logs safely.

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
IMAGE_SIZE     = (320, 320)
BATCH_SIZE     = 16
EPOCHS         = 64       # good balance
LEARNING_RATE  = 1e-3     # 2x LR for every 2x in batch size
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## 2. Dataset Instantiation

Re-instantiates the `tf.data.Dataset` objects from the partitioned directories, applying the new 384x384 resolution and the adjusted batch size whilst maintaining categorical label encoding.

In [ ]:
# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

train_ds  = train_ds.cache().prefetch(AUTOTUNE)
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## Config

## 2. Architecture Configuration

Defines the specific filter counts, kernel sizes, and strides for the residual blocks (`conv_setup`), alongside the dense layer dimensions (`dense_setup`), and loads the previously serialised class weights from a JSON file to address the significant dataset imbalance during the loss calculation.

In [ ]:
# ── Custom CNN architecture config ───────────────────────────────────────────
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [512, 256]

# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}

# Step 3.5: Model Compilation and Lifecycle Management

In [7]:
model = MyCNN(
    augmentation_layer=None,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    num_classes=N_CLASSES,
)

## 1. Metrics and Loss Formulation

Instantiates fresh, stateful metrics for the model, featuring the Macro **F1-score** (`tfa.metrics.F1Score`) alongside Categorical Accuracy and AUC to ensure a balanced evaluation across all 23 classes, and compiles the model using `CategoricalCrossentropy` with a `label_smoothing` factor of 0.1 to penalise overconfidence in predictions.

In [8]:
def make_metrics(num_classes):
    """Fresh metric instances per model — metrics are stateful and must not be shared."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]

In [10]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=SGD(learning_rate=LEARNING_RATE, name="optimizer", decay=0.01), 
    metrics=make_metrics(num_classes=N_CLASSES)
)

## 2. Callback Configuration

Implements a custom cosine annealing schedule with a linear warmup phase to prevent outsized gradient updates during the initial epochs, given the relatively high base learning rate (1e-1), and configures an `EarlyStopping` callback monitoring validation loss with a patience of 7 epochs, ensuring the best weights are automatically restored to prevent unnecessary computational expenditure.

In [9]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Linear warmup then cosine annealing.

    Warmup matters especially for the larger LR used with MyCNN (2e-3):
    without it, the first few batches produce outsized gradient updates.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Compile and prepare callbacks

In [10]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=tfa.optimizers.AdamW(learning_rate=LEARNING_RATE, name="optimizer", weight_decay=1e-7), 
    metrics=make_metrics(num_classes=N_CLASSES)
)

In [11]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    checkpoints_folder_path / f"checkpoint_{model.name}",
    save_best_only=True,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_folder_path / f"metric_{model.name}.csv")

In [12]:
lr_scheduler_callback = LearningRateScheduler(make_cosine_warmup_scheduler(LEARNING_RATE, EPOCHS, warmup_epochs=3))

In [13]:
# EarlyStopping: stops training if val_loss doesn't improve for "patience" epochs
# and restores the best weights automatically
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)


In [14]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback,
    early_stopping_callback
]

# Step 3.6: Execution and Simple Evaluation

## 1. Training Loop

Executes the `model.fit()` routine using the training dataset, passing the pre-calculated class weights to handle the significant dataset imbalance.

*Utilises `clear_session()` immediately after training and evaluation to flush the GPU memory and prevent state leakage into subsequent model experiments.*

## 2. Performance Evaluation

Evaluates the restored best weights against the unseen `test_ds`, returning a dictionary of metrics to benchmark the model's generalisation capabilities.

In [15]:
# Train the model
model_fit_data = model.fit(
    train_ds,
    validation_data=val_ds,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)
model_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
# Limpa a memória da GPU/RAM ocupada pelo modelo que acabou de treinar
clear_session()

model_fit_data, model_eval_data

Epoch 1/64
583/583 [==============================] - ETA: 0s - loss: 2.9261 - accuracy: 0.1629 - auc: 0.6970 - f1_score: 0.1449

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


583/583 [==============================] - 137s 193ms/step - loss: 2.9261 - accuracy: 0.1629 - auc: 0.6970 - f1_score: 0.1449 - val_loss: 2.6598 - val_accuracy: 0.2485 - val_auc: 0.7969 - val_f1_score: 0.2095 - lr: 3.3333e-04
Epoch 2/64
583/583 [==============================] - 162s 277ms/step - loss: 2.7813 - accuracy: 0.2030 - auc: 0.7469 - f1_score: 0.1802 - val_loss: 2.7607 - val_accuracy: 0.2204 - val_auc: 0.7877 - val_f1_score: 0.1946 - lr: 6.6667e-04
Epoch 3/64
583/583 [==============================] - 170s 291ms/step - loss: 2.6829 - accuracy: 0.2308 - auc: 0.7780 - f1_score: 0.2110 - val_loss: 3.0548 - val_accuracy: 0.1812 - val_auc: 0.7158 - val_f1_score: 0.1244 - lr: 0.0010
Epoch 4/64
583/583 [==============================] - 154s 264ms/step - loss: 2.5684 - accuracy: 0.2629 - auc: 0.8085 - f1_score: 0.2419 - val_loss: 3.1854 - val_accuracy: 0.1893 - val_auc: 0.7321 - val_f1_score: 0.1554 - lr: 0.0010
Epoch 5/64
583/583 [==============================] - ETA: 0s - loss: 2

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


...

583/583 [==============================] - 161s 276ms/step - loss: 1.6886 - accuracy: 0.5861 - auc: 0.9497 - f1_score: 0.5817 - val_loss: 1.9023 - val_accuracy: 0.5281 - val_auc: 0.9280 - val_f1_score: 0.5045 - lr: 8.3964e-04
Epoch 21/64
583/583 [==============================] - 146s 251ms/step - loss: 1.6565 - accuracy: 0.5937 - auc: 0.9536 - f1_score: 0.5904 - val_loss: 1.9819 - val_accuracy: 0.5156 - val_auc: 0.9198 - val_f1_score: 0.4866 - lr: 8.2030e-04
Epoch 22/64
583/583 [==============================] - 156s 267ms/step - loss: 1.6004 - accuracy: 0.6169 - auc: 0.9577 - f1_score: 0.6155 - val_loss: 1.9098 - val_accuracy: 0.5326 - val_auc: 0.9250 - val_f1_score: 0.5162 - lr: 8.0011e-04
Epoch 23/64
583/583 [==============================] - ETA: 0s - loss: 1.5456 - accuracy: 0.6420 - auc: 0.9624 - f1_score: 0.6410

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


583/583 [==============================] - 147s 252ms/step - loss: 1.5456 - accuracy: 0.6420 - auc: 0.9624 - f1_score: 0.6410 - val_loss: 1.8584 - val_accuracy: 0.5572 - val_auc: 0.9308 - val_f1_score: 0.5375 - lr: 7.7912e-04
Epoch 24/64
583/583 [==============================] - 141s 242ms/step - loss: 1.4904 - accuracy: 0.6596 - auc: 0.9669 - f1_score: 0.6591 - val_loss: 1.8615 - val_accuracy: 0.5517 - val_auc: 0.9297 - val_f1_score: 0.5283 - lr: 7.5740e-04
Epoch 25/64
583/583 [==============================] - 126s 216ms/step - loss: 1.4323 - accuracy: 0.6825 - auc: 0.9713 - f1_score: 0.6838 - val_loss: 1.8905 - val_accuracy: 0.5502 - val_auc: 0.9267 - val_f1_score: 0.5286 - lr: 7.3499e-04
Epoch 26/64
583/583 [==============================] - 156s 267ms/step - loss: 1.3866 - accuracy: 0.7013 - auc: 0.9742 - f1_score: 0.7031 - val_loss: 1.9135 - val_accuracy: 0.5296 - val_auc: 0.9266 - val_f1_score: 0.5098 - lr: 7.1196e-04
Epoch 27/64
583/583 [==============================] - 111s 

(<keras.callbacks.History at 0x20e1b06fc40>,
 {'loss': 1.8360590934753418,
  'accuracy': 0.5544015765190125,
  'auc': 0.9340656399726868,
  'f1_score': 0.5343406200408936})

The curves are now telling a completely different and much more readable story — this is genuine overfitting, which is actually a good sign. It means the model has enough capacity and is learning real features, just memorising the training set rather than generalising. The previous problem (val > train) was the model being unable to learn at all; this problem (train >> val) is the normal problem that regularisation fixes.
Concretely: training accuracy linearly reaches ~0.75 while validation plateaus at ~0.55 and starts drifting upward in loss from around epoch 20. That ~20 percentage point train/val gap is the target to close.

## What's causing it

We swung from too much regularisation (strong Mixup + strong augmentation + dropout 0.5) all the way to almost none (no augmentation, no Mixup, weight_decay=1e-7 which is essentially zero). The model now fits the training set comfortably but hasn't seen enough variation to generalise well.

## What to do, in order of impact

1. Add augmentation back — lightly. 
2. Increase weight decay to a meaningful value (for instance 1e-4).
3. Increase dropout slightly (from 0.3 to 0.4).
4. Add Mixup back, but very mildly (alpha=0.1 instead of alpha=0.4).
5. Extend training with more patience.